# Qwen NER GRPO Notebook

1. 环境与路径配置
2. 数据加载与抽样检查
3. 基线推理
4. 预测解析与 reward 设计
5. dev 小批量离线检查
6. GRPO 训练准备
7. GRPO 训练入口


In [1]:
from pathlib import Path
import json
import re
from typing import Dict, List, Tuple

PROJECT_ROOT = Path.cwd()

# 现在的训练链路是：
# base model -> SFT -> GRPO
#
# 所以这里不再直接从原始 Qwen 基座模型开始做 GRPO，
# 而是优先从 SFT 训练后的目录继续往下训练。
BASE_MODEL_PATH = Path(r"C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B-Instruct")
SFT_MODEL_PATH = PROJECT_ROOT / "sft_runs" / "qwen_ner_sft_train_final"
MODEL_PATH = SFT_MODEL_PATH if SFT_MODEL_PATH.exists() else BASE_MODEL_PATH
DATA_DIR = PROJECT_ROOT / "llm_ner_dataset2" / "output"
TRAIN_PATH = DATA_DIR / "train.json"
DEV_PATH = DATA_DIR / "dev.json"
TEST_PATH = DATA_DIR / "test.json"
RUN_DIR = PROJECT_ROOT / "grpo_runs"
RUN_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_MODEL_PATH exists:", BASE_MODEL_PATH.exists())
print("SFT_MODEL_PATH exists:", SFT_MODEL_PATH.exists())
print("MODEL_PATH:", MODEL_PATH)
print("MODEL_PATH exists:", MODEL_PATH.exists())
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("DEV_PATH exists:", DEV_PATH.exists())
print("TEST_PATH exists:", TEST_PATH.exists())


PROJECT_ROOT: D:\github\Reinforcement-Learning\GRPO
BASE_MODEL_PATH exists: True
SFT_MODEL_PATH exists: False
MODEL_PATH: C:\Users\lihaodong\.cache\modelscope\hub\models\Qwen\Qwen2___5-0___5B-Instruct
MODEL_PATH exists: True
TRAIN_PATH exists: True
DEV_PATH exists: True
TEST_PATH exists: True


In [2]:
# 优先使用 datasets 读取，后面如果要接 Trainer 会更顺手。
# 但如果本地环境因为 numpy / pandas 兼容问题导致 datasets 不能导入，
# 就自动退回到手写 jsonl 读取，保证 notebook 至少能继续往下跑。

def load_jsonl(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

try:
    from datasets import load_dataset

    data_files = {
        "train": str(TRAIN_PATH),
        "dev": str(DEV_PATH),
        "test": str(TEST_PATH),
    }

    dataset_dict = load_dataset("json", data_files=data_files)
    train_data = dataset_dict["train"]
    dev_data = dataset_dict["dev"]
    test_data = dataset_dict["test"]
    data_backend = "datasets"
    print(dataset_dict)
except Exception as e:
    train_data = load_jsonl(TRAIN_PATH)
    dev_data = load_jsonl(DEV_PATH)
    test_data = load_jsonl(TEST_PATH)
    data_backend = "python_list"
    print("datasets 不可用，已退回手写 jsonl 加载：", type(e).__name__, str(e)[:200])

print("data_backend:", data_backend)
print("train:", len(train_data))
print("dev:", len(dev_data))
print("test:", len(test_data))
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))


D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 18000
    })
    dev: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 3496
    })
    test: Dataset({
        features: ['prompt', 'answer', 'schema', 'text'],
        num_rows: 2686
    })
})
data_backend: datasets
train: 18000
dev: 3496
test: 2686
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [
      "浙商银行"
    ],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 这里优先加载 SFT 之后的模型目录。
# 如果 SFT 目录不存在，才回退到原始 base model。
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)

sample = train_data[0]
chat_text = tokenizer.apply_chat_template(
    sample["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)

model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
    )

completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
response = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

print(chat_text)
print("text:", sample["text"])
print("schema:", sample["schema"])
print("gold:", json.dumps(sample["answer"], ensure_ascii=False))
print("pred:", response)


D:\anaconda\install\envs\Vocal_Remover\lib\site-packages\librosa\util\files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1007.56it/s]


<|im_start|>system
你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。<|im_end|>
<|im_start|>user
请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。

schema: ["address", "book", "company", "game", "government", "movie"]
text: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，<|im_end|>
<|im_start|>assistant

text: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": [], "book": [], "company": ["浙商银行"], "game": [], "government": [], "movie": []}
pred: ["浙商银行企业信贷部", "叶老桂博士"]


In [4]:
def normalize_answer(answer_obj: dict, schema: List[str]) -> Dict[str, List[str]]:
    normalized = {}
    for key in schema:
        value = answer_obj.get(key, []) if isinstance(answer_obj, dict) else []
        if value is None:
            value = []
        elif not isinstance(value, list):
            value = [value]
        value = [str(x).strip() for x in value if str(x).strip()]
        normalized[key] = sorted(list(dict.fromkeys(value)))
    return normalized


def extract_json_object(text: str) -> str:
    text = text.strip()
    match = re.search(r"\{.*\}", text, flags=re.S)
    return match.group(0) if match else ""


def parse_prediction(text: str, schema: List[str]) -> Tuple[Dict[str, List[str]], dict]:
    meta = {
        "json_parse_ok": False,
        "is_dict_object": False,
        "has_extra_keys": False,
        "has_missing_keys": False,
    }

    raw = text.strip()
    candidate = raw

    try:
        parsed = json.loads(candidate)
        meta["json_parse_ok"] = True
    except Exception:
        candidate = extract_json_object(raw)
        if candidate:
            try:
                parsed = json.loads(candidate)
                meta["json_parse_ok"] = True
            except Exception:
                parsed = {}
        else:
            parsed = {}

    if isinstance(parsed, dict):
        meta["is_dict_object"] = True
    else:
        parsed = {}

    extra_keys = set(parsed.keys()) - set(schema)
    missing_keys = set(schema) - set(parsed.keys())
    meta["has_extra_keys"] = bool(extra_keys)
    meta["has_missing_keys"] = bool(missing_keys)

    return normalize_answer(parsed, schema), meta


def to_entity_pairs(answer: Dict[str, List[str]], schema: List[str]) -> set:
    pairs = set()
    for key in schema:
        values = answer.get(key, [])
        if values is None:
            continue
        if not isinstance(values, list):
            values = [values]
        for value in values:
            value = str(value).strip()
            if value:
                pairs.add((key, value))
    return pairs


def f1_score(pred_pairs, gold_pairs) -> float:
    pred_set = set(pred_pairs)
    gold_set = set(gold_pairs)

    # 整条都空，只说明“没有乱抽”，只给低保底分
    if len(gold_set) == 0 and len(pred_set) == 0:
        return 0.2
    if len(gold_set) == 0 and len(pred_set) > 0:
        return 0.0
    if len(pred_set) == 0 and len(gold_set) > 0:
        return 0.0

    tp = len(pred_set & gold_set)
    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def compute_reward(pred_text: str, gold_answer: Dict[str, List[str]], schema: List[str]) -> Tuple[float, dict]:
    pred_answer, meta = parse_prediction(pred_text, schema)

    r_format = 1.0 if (meta["json_parse_ok"] and meta["is_dict_object"]) else 0.0
    r_schema = 1.0 if (not meta["has_extra_keys"] and not meta["has_missing_keys"]) else 0.0

    # 只有过了格式门槛，才允许进入实体能力评分
    pred_pairs = to_entity_pairs(pred_answer, schema)
    gold_pairs = to_entity_pairs(gold_answer, schema)
    if meta["json_parse_ok"] and meta["is_dict_object"]:
        r_entity = f1_score(pred_pairs, gold_pairs)
    else:
        r_entity = 0.0

    r_penalty = 1.0 if len(gold_pairs) > 0 and len(pred_pairs) == 0 else 0.0

    reward = 0.2 * r_format + 0.2 * r_schema + 0.8 * r_entity - 0.2 * r_penalty

    return reward, {
        "pred_answer": pred_answer,
        "r_format": r_format,
        "r_schema": r_schema,
        "r_entity": r_entity,
        "r_penalty": r_penalty,
        "pred_pairs": sorted(list(pred_pairs)),
        "gold_pairs": sorted(list(gold_pairs)),
        **meta,
    }


reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])
print("reward:", reward)
print(json.dumps(reward_meta, ensure_ascii=False, indent=2))


reward: -0.2
{
  "pred_answer": {
    "address": [],
    "book": [],
    "company": [],
    "game": [],
    "government": [],
    "movie": []
  },
  "r_format": 0.0,
  "r_schema": 0.0,
  "r_entity": 0.0,
  "r_penalty": 1.0,
  "pred_pairs": [],
  "gold_pairs": [
    [
      "company",
      "浙商银行"
    ]
  ],
  "json_parse_ok": true,
  "is_dict_object": false,
  "has_extra_keys": false,
  "has_missing_keys": true
}


In [5]:
# 用 dev 集做一个小批量离线打分测试
# 注意：如果 dev_data 是 datasets.Dataset，直接写 dev_data[:50] 会得到按列组织的数据，
# 不是一条条样本。所以这里统一按索引逐条取样本。

subset_size = min(50, len(dev_data))
results = []

with torch.no_grad():
    for i in range(subset_size):
        sample = dev_data[i]

        chat_text = tokenizer.apply_chat_template(
            sample["prompt"],
            tokenize=False,
            add_generation_prompt=True,
        )

        model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=128,
            do_sample=False,
        )

        completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
        response = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)[0]

        reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])

        results.append({
            "index": i,
            "text": sample["text"],
            "schema": sample["schema"],
            "gold_answer": sample["answer"],
            "response": response,
            "reward": reward,
            "reward_meta": reward_meta,
        })

        if i < 3:
            print(f"===== sample {i} =====")
            print("text:", sample["text"])
            print("schema:", sample["schema"])
            print("gold:", json.dumps(sample["answer"], ensure_ascii=False))
            print("response:", response)
            print("reward:", reward)
            print(json.dumps(reward_meta, ensure_ascii=False, indent=2))
            print()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


===== sample 0 =====
text: 09科特布斯vs斯图加特
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": [], "book": [], "company": [], "game": [], "government": [], "movie": []}
response: ["09科特布斯", "斯图加特"]
reward: 0.0
{
  "pred_answer": {
    "address": [],
    "book": [],
    "company": [],
    "game": [],
    "government": [],
    "movie": []
  },
  "r_format": 0.0,
  "r_schema": 0.0,
  "r_entity": 0.0,
  "r_penalty": 0.0,
  "pred_pairs": [],
  "gold_pairs": [],
  "json_parse_ok": true,
  "is_dict_object": false,
  "has_extra_keys": false,
  "has_missing_keys": true
}

===== sample 1 =====
text: 09科特布斯vs斯图加特
schema: ['name', 'organization', 'position', 'scene']
gold: {"name": [], "organization": ["斯图加特", "科特布斯"], "position": [], "scene": []}
response: ["09科特布斯", "斯图加特"]
reward: -0.2
{
  "pred_answer": {
    "name": [],
    "organization": [],
    "position": [],
    "scene": []
  },
  "r_format": 0.0,
  "r_schema": 0.0,
  "r_entity": 0.0,
  "r_penalty": 1.0,


In [6]:
# 统计这批 dev 样本的整体情况

avg_reward = sum(x["reward"] for x in results) / len(results)
avg_r_entity = sum(x["reward_meta"]["r_entity"] for x in results) / len(results)

json_ok_rate = sum(1 for x in results if x["reward_meta"]["json_parse_ok"]) / len(results)
dict_ok_rate = sum(1 for x in results if x["reward_meta"]["is_dict_object"]) / len(results)
extra_key_rate = sum(1 for x in results if x["reward_meta"]["has_extra_keys"]) / len(results)
missing_key_rate = sum(1 for x in results if x["reward_meta"]["has_missing_keys"]) / len(results)

print("样本数:", len(results))
print("avg_reward:", round(avg_reward, 4))
print("avg_r_entity:", round(avg_r_entity, 4))
print("json_ok_rate:", round(json_ok_rate, 4))
print("dict_ok_rate:", round(dict_ok_rate, 4))
print("extra_key_rate:", round(extra_key_rate, 4))
print("missing_key_rate:", round(missing_key_rate, 4))


样本数: 50
avg_reward: -0.12
avg_r_entity: 0.0
json_ok_rate: 0.66
dict_ok_rate: 0.34
extra_key_rate: 0.0
missing_key_rate: 1.0


In [7]:
# 看 reward 最低的几个样本，检查“低分是否低得有道理”

sorted_results = sorted(results, key=lambda x: x["reward"])

for item in sorted_results[:5]:
    print("===== bad case =====")
    print("index:", item["index"])
    print("text:", item["text"])
    print("schema:", item["schema"])
    print("gold:", json.dumps(item["gold_answer"], ensure_ascii=False))
    print("response:", item["response"])
    print("reward:", item["reward"])
    print(json.dumps(item["reward_meta"], ensure_ascii=False, indent=2))
    print()


===== bad case =====
index: 1
text: 09科特布斯vs斯图加特
schema: ['name', 'organization', 'position', 'scene']
gold: {"name": [], "organization": ["斯图加特", "科特布斯"], "position": [], "scene": []}
response: ["09科特布斯", "斯图加特"]
reward: -0.2
{
  "pred_answer": {
    "name": [],
    "organization": [],
    "position": [],
    "scene": []
  },
  "r_format": 0.0,
  "r_schema": 0.0,
  "r_entity": 0.0,
  "r_penalty": 1.0,
  "pred_pairs": [],
  "gold_pairs": [
    [
      "organization",
      "斯图加特"
    ],
    [
      "organization",
      "科特布斯"
    ]
  ],
  "json_parse_ok": true,
  "is_dict_object": false,
  "has_extra_keys": false,
  "has_missing_keys": true
}

===== bad case =====
index: 2
text: 这也是我们人居环境委员会从住区向城镇发展的一个非常重要的标志。在这个体系里面，
schema: ['address', 'book', 'company', 'game', 'government', 'movie']
gold: {"address": [], "book": [], "company": [], "game": [], "government": ["人居环境委员会"], "movie": []}
response: ["住区", "城镇"]
reward: -0.2
{
  "pred_answer": {
    "address": [],
    "book": [],
    "com

## 训练前准备

在真正开始 GRPO 训练前，先确认依赖和环境。

如果下面的依赖检查发现缺包，先在终端里安装：

```bash
pip install -U trl peft datasets
```

如果 `datasets` 仍然报 NumPy / pandas 兼容错误，优先处理环境问题，再继续训练。


In [8]:
# 检查训练所需依赖是否可用
required_modules = ["trl", "peft", "datasets", "accelerate", "transformers"]
dep_status = {}

for mod_name in required_modules:
    try:
        mod = __import__(mod_name)
        dep_status[mod_name] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        dep_status[mod_name] = f"NOT_AVAILABLE: {type(e).__name__}: {str(e)[:120]}"

print(json.dumps(dep_status, ensure_ascii=False, indent=2))


{
  "trl": "1.0.0",
  "peft": "0.18.1",
  "datasets": "4.8.4",
  "accelerate": "1.12.0",
  "transformers": "5.4.0"
}


In [9]:
# 准备一个更适合后面 GRPOTrainer 使用的训练子集。
# 先只取一小部分做冒烟测试，避免一上来就全量训练。

train_subset_size = min(50, len(train_data))
eval_subset_size = min(10, len(dev_data))

grpo_train_samples = [train_data[i] for i in range(train_subset_size)]
grpo_eval_samples = [dev_data[i] for i in range(eval_subset_size)]

print("train subset:", len(grpo_train_samples))
print("eval subset:", len(grpo_eval_samples))
print(json.dumps(grpo_train_samples[0], ensure_ascii=False, indent=2))


train subset: 50
eval subset: 10
{
  "prompt": [
    {
      "role": "system",
      "content": "你是专门进行实体抽取的专家。请严格按照 JSON 格式回答，不要输出额外解释。"
    },
    {
      "role": "user",
      "content": "请从文本text中抽取出符合 schema 定义的实体，不存在的实体类型返回空列表。\n\nschema: [\"address\", \"book\", \"company\", \"game\", \"government\", \"movie\"]\ntext: 浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
    }
  ],
  "answer": {
    "address": [],
    "book": [],
    "company": [
      "浙商银行"
    ],
    "game": [],
    "government": [],
    "movie": []
  },
  "schema": [
    "address",
    "book",
    "company",
    "game",
    "government",
    "movie"
  ],
  "text": "浙商银行企业信贷部叶老桂博士则从另一个角度对五道门槛进行了解读。叶老桂认为，对目前国内商业银行而言，"
}


In [10]:
# 把当前的 compute_reward 包装成更接近 TRL reward_funcs 的形式。
# 这样后面如果接 GRPOTrainer，就不需要大改 reward 主逻辑。

def completion_to_text(completion):
    # completion 可能是：
    # 1. 普通字符串
    # 2. 对话消息列表，例如 [{"role": "assistant", "content": "..."}]
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict) and "content" in item:
                parts.append(str(item["content"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)

    return str(completion)


def ner_reward_func(completions, answer, schema, **kwargs):
    rewards = []
    for completion, gold_answer, one_schema in zip(completions, answer, schema):
        pred_text = completion_to_text(completion)
        reward, _ = compute_reward(pred_text, gold_answer, one_schema)
        rewards.append(reward)
    return rewards


# 先做一个最小测试，确认 wrapper 行为和 compute_reward 一致
test_rewards = ner_reward_func(
    completions=[response],
    answer=[sample["answer"]],
    schema=[sample["schema"]],
)
print(test_rewards)


[-0.2]


In [11]:
# 这一格只负责准备训练对象，不会自动开跑。
# 如果依赖没装好，会直接报清楚，不继续往下走。

from importlib.util import find_spec

assert find_spec("trl") is not None, "缺少 trl，请先安装：pip install -U trl"
assert find_spec("peft") is not None, "缺少 peft，请先安装：pip install -U peft"

from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

# LoRA 先走保守配置，目的是先把训练跑通
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 训练参数先压低，只做冒烟测试
training_args = GRPOConfig(
    output_dir=str(RUN_DIR / "grpo_qwen_ner_smoke"),
    per_device_train_batch_size=1, # GRPO 内部会自动累积到一个更合理的 batch_size，所以这里先设小，避免显存不够
    gradient_accumulation_steps=8,# 4 步累积成一个有效 batch_size 8
    learning_rate=1e-6,# 先用一个非常小的学习率，避免一上来就大幅破坏模型能力，导致 reward 崩溃
    num_train_epochs=1,# 先只跑 1 个 epoch，确认能正常更新参数和提升 reward，再考虑增加 epoch 数
    logging_steps=1,# 每条样本都 log，方便观察 reward 变化趋势
    save_steps=200,# 每 200 步保存一次模型，先不要频繁保存，避免磁盘压力过大
    max_completion_length=128,# 生成长度先限制在 128，避免过长输出导致的显存问题和训练不稳定
    num_generations=4,# 每条样本生成 4 个不同的 completion，增加 reward 信号的多样性，帮助训练更稳定
    bf16=torch.cuda.is_available(),# 如果有支持的 GPU，优先使用 bf16 加速训练；如果没有，自动退回到 fp16 或 fp32
    report_to=[],# 先不接任何第三方日志平台，所有日志都输出到控制台，方便观察和调试
    top_k= 50,
    top_p= 0.95,

)

print(training_args)


GRPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.0,
bf16=True,
bf16_full_eval=False,
cache_implementation=None,
cast_lm_head_to_fp32=False,
chat_template_kwargs=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
delta=None,
disable_dropout=False,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
ds3_gather_for_generation=True,
enable_jit_checkpoint=False,
epsilon=0.2,
epsilon_hig

In [12]:
# 构造 GRPOTrainer。
# 这里的 model 要从 SFT 模型目录继续，而不是重新从 base model 开始。
# 这样 GRPO 的起点已经具备基本的 JSON 格式输出能力。

trainer = GRPOTrainer(
    model=str(MODEL_PATH),
    reward_funcs=ner_reward_func, # 直接传函数对象，GRPOTrainer 内部会自动处理批量输入输出，不需要我们改 compute_reward 的逻辑
    args=training_args,
    train_dataset=grpo_train_samples,
    eval_dataset=grpo_eval_samples,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(type(trainer))
print("trainer ready")


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1018.25it/s]


<class 'trl.trainer.grpo_trainer.GRPOTrainer'>
trainer ready


In [13]:
# 真正开始训练前，建议先确认：
# 1. dep-check 没有缺包
# 2. dev-check 的 reward 合理
# 3. trainer-build 可以正常创建 trainer
#
# 确认没问题后，再解除下面两行注释开始冒烟训练。

train_result = trainer.train()
print(train_result)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.000000
2,-0.198049
3,0.000000
4,0.101879
5,-0.184988
6,-0.143988
7,0.000000
8,-0.123209
9,-0.330247
10,-0.145953


TrainOutput(global_step=25, training_loss=-0.046875752210617065, metrics={'train_runtime': 78.9189, 'train_samples_per_second': 0.634, 'train_steps_per_second': 0.317, 'total_flos': 0.0, 'train_loss': -0.046875752210617065})


In [14]:
for row in trainer.state.log_history[-20:]:
  print(row)


{'loss': 0.0, 'grad_norm': 0.0, 'learning_rate': 7.599999999999999e-07, 'num_tokens': 7133.0, 'completions/mean_length': 17.75, 'completions/min_length': 9.0, 'completions/max_length': 29.0, 'completions/clipped_ratio': 0.0, 'completions/mean_terminated_length': 17.75, 'completions/min_terminated_length': 9.0, 'completions/max_terminated_length': 29.0, 'rewards/ner_reward_func/mean': -0.10000000149011612, 'rewards/ner_reward_func/std': 0.10690450668334961, 'reward': -0.10000000149011612, 'reward_std': 0.10690449923276901, 'frac_reward_zero_std': 1.0, 'entropy': 0.7258031442761421, 'clip_ratio/low_mean': 0.0, 'clip_ratio/low_min': 0.0, 'clip_ratio/high_mean': 0.0, 'clip_ratio/high_max': 0.0, 'clip_ratio/region_mean': 0.0, 'step_time': 2.054476599674672, 'epoch': 0.28, 'step': 7}
{'loss': -0.12320924550294876, 'grad_norm': 0.18248489499092102, 'learning_rate': 7.2e-07, 'num_tokens': 8197.0, 'completions/mean_length': 27.5, 'completions/min_length': 7.0, 'completions/max_length': 62.0, 'c

In [18]:
def inspect_group_generations(sample, model_obj, tokenizer_obj, n=8, max_new_tokens=128):
    chat_text = tokenizer_obj.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer_obj([chat_text], return_tensors="pt").to(model_obj.device)

    with torch.no_grad():
        generated_ids = model_obj.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            top_k=50,
            num_return_sequences=n,
            repetition_penalty=1.05,
        )

    prompt_len = model_inputs.input_ids.shape[1]
    completion_ids = generated_ids[:, prompt_len:]
    responses = tokenizer_obj.batch_decode(completion_ids, skip_special_tokens=True)

    rows = []
    for i, resp in enumerate(responses):
        reward, meta = compute_reward(resp, sample["answer"], sample["schema"])
        rows.append({
            "idx": i,
            "response": resp,
            "reward": reward,
            "pred_answer": meta["pred_answer"],
            "pred_pairs": meta["pred_pairs"],
            "gold_pairs": meta["gold_pairs"],
            "json_parse_ok": meta["json_parse_ok"],
            "r_entity": meta["r_entity"],
        })

    return rows
rows = inspect_group_generations(train_data[0], model, tokenizer, n=8)

for row in rows:
    print("=" * 80)
    print("idx:", row["idx"])
    print("reward:", row["reward"])
    print("r_entity:", row["r_entity"])
    print("json_parse_ok:", row["json_parse_ok"])
    print("pred_pairs:", row["pred_pairs"])
    print("response:", row["response"])

idx: 0
reward: -0.2
r_entity: 0.0
json_parse_ok: False
pred_pairs: []
response: [
  {
    "entity": "浙商银行",
    "type": "entity_type"
  },
  {
    "entity": "叶老桂博士",
    "type": "person"
  },
  {
    "entity": "五道门槛",
    "type": "resource"
  }
]
{
  "entity": "浙江",
  "type": "location"
}
idx: 1
reward: -0.2
r_entity: 0.0
json_parse_ok: True
pred_pairs: []
response: [
    {
        "entity": "浙商银行企业信贷部",
        "type": "组织单位"
    },
    {
        "entity": "叶老桂博士",
        "type": "个人"
    }
]
idx: 2
reward: -0.2
r_entity: 0.0
json_parse_ok: True
pred_pairs: []
response: [
  {
    "entity": "浙商银行企业信贷部",
    "type": "ORGANIZATION"
  },
  {
    "entity": "叶老桂博士",
    "type": "PERSON"
  }
]
idx: 3
reward: -0.2
r_entity: 0.0
json_parse_ok: True
pred_pairs: []
response: ["浙商银行企业信贷部"]
idx: 4
reward: -0.2
r_entity: 0.0
json_parse_ok: True
pred_pairs: []
response: ["浙商银行企业信贷部", "叶老桂博士", "五道门槛"]
idx: 5
reward: -0.2
r_entity: 0.0
json_parse_ok: True
pred_pairs: []
response: ["浙商银行企业信贷部", "叶老桂博士

## 训练后评估

下面几格用于训练完成后的离线评估。

评估思路：

1. 让模型在 dev/test 上逐条生成
2. 用当前同一套 `compute_reward` 做自动评分
3. 额外统计全局实体对层面的 micro precision / recall / F1

这样你可以同时看：
- 输出格式是否稳定
- reward 是否提升
- 真正的实体抽取能力是否提升


In [ ]:
def run_single_inference(sample, model_obj, tokenizer_obj, max_new_tokens=128, do_sample=False):
    chat_text = tokenizer_obj.apply_chat_template(
        sample["prompt"],
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer_obj([chat_text], return_tensors="pt").to(model_obj.device)

    with torch.no_grad():
        generated_ids = model_obj.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

    completion_ids = generated_ids[:, model_inputs.input_ids.shape[1]:]
    response = tokenizer_obj.batch_decode(completion_ids, skip_special_tokens=True)[0]
    return response


def pair_metrics(pred_pair_sets, gold_pair_sets):
    total_tp = 0
    total_pred = 0
    total_gold = 0

    for pred_pairs, gold_pairs in zip(pred_pair_sets, gold_pair_sets):
        pred_set = set(pred_pairs)
        gold_set = set(gold_pairs)
        total_tp += len(pred_set & gold_set)
        total_pred += len(pred_set)
        total_gold += len(gold_set)

    precision = total_tp / total_pred if total_pred > 0 else 0.0
    recall = total_tp / total_gold if total_gold > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": total_tp,
        "pred_total": total_pred,
        "gold_total": total_gold,
    }


def evaluate_split(split_data, model_obj, tokenizer_obj, max_samples=100, max_new_tokens=128):
    eval_size = min(max_samples, len(split_data))
    eval_rows = []
    pred_pair_sets = []
    gold_pair_sets = []

    for i in range(eval_size):
        sample = split_data[i]
        response = run_single_inference(sample, model_obj, tokenizer_obj, max_new_tokens=max_new_tokens, do_sample=False)
        reward, reward_meta = compute_reward(response, sample["answer"], sample["schema"])

        eval_rows.append({
            "index": i,
            "text": sample["text"],
            "schema": sample["schema"],
            "gold_answer": sample["answer"],
            "response": response,
            "reward": reward,
            "reward_meta": reward_meta,
        })

        pred_pair_sets.append(reward_meta["pred_pairs"])
        gold_pair_sets.append(reward_meta["gold_pairs"])

    summary = {
        "num_samples": len(eval_rows),
        "avg_reward": sum(x["reward"] for x in eval_rows) / len(eval_rows),
        "avg_r_entity": sum(x["reward_meta"]["r_entity"] for x in eval_rows) / len(eval_rows),
        "json_ok_rate": sum(1 for x in eval_rows if x["reward_meta"]["json_parse_ok"]) / len(eval_rows),
        "dict_ok_rate": sum(1 for x in eval_rows if x["reward_meta"]["is_dict_object"]) / len(eval_rows),
        "schema_strict_rate": sum(1 for x in eval_rows if x["reward_meta"]["r_schema"] == 1.0) / len(eval_rows),
    }

    summary.update(pair_metrics(pred_pair_sets, gold_pair_sets))
    return eval_rows, summary


In [ ]:
# 训练前或训练后都可以先跑这个单元。
# 默认还是用当前内存里的 model。
# 如果后面你加载了 LoRA 或训练后的模型对象，直接把 model 换成对应对象即可。

dev_eval_rows, dev_eval_summary = evaluate_split(
    split_data=dev_data,
    model_obj=model,
    tokenizer_obj=tokenizer,
    max_samples=100,
    max_new_tokens=128,
)

print(json.dumps(dev_eval_summary, ensure_ascii=False, indent=2))


In [ ]:
# 查看评估里最差的几个案例，帮助判断模型到底错在哪里

bad_eval_rows = sorted(dev_eval_rows, key=lambda x: x["reward"])[:5]

for row in bad_eval_rows:
    print("===== eval bad case =====")
    print("index:", row["index"])
    print("text:", row["text"])
    print("schema:", row["schema"])
    print("gold:", json.dumps(row["gold_answer"], ensure_ascii=False))
    print("response:", row["response"])
    print("reward:", row["reward"])
    print(json.dumps(row["reward_meta"], ensure_ascii=False, indent=2))
    print()


In [ ]:
# 单条推理演示：你可以随时替换 example_idx 看任意一条样本

example_idx = 0
demo_sample = test_data[example_idx]
demo_response = run_single_inference(demo_sample, model, tokenizer, max_new_tokens=128, do_sample=False)
demo_reward, demo_meta = compute_reward(demo_response, demo_sample["answer"], demo_sample["schema"])

print("text:", demo_sample["text"])
print("schema:", demo_sample["schema"])
print("gold:", json.dumps(demo_sample["answer"], ensure_ascii=False))
print("response:", demo_response)
print("reward:", demo_reward)
print(json.dumps(demo_meta, ensure_ascii=False, indent=2))
